<div align="center">

## Zadanie 1

</div>

In [21]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.cluster import KMeans

# Załadowanie danych
digits = load_digits()
images = digits.data
targets = digits.target

def get_semi_targets(targets, labeled_ratio = 0.05):
    # Losujemy indeksy, które zostaną etykietowane
    n_samples = len(targets)
    n_labeled = int(n_samples * labeled_ratio)

    rng = np.random.default_rng(seed=42)  # dla powtarzalności
    labeled_indices = rng.choice(n_samples, size=n_labeled, replace=False)

    semi_targets = np.full_like(targets, fill_value=-1)

    semi_targets[labeled_indices] = targets[labeled_indices]

    print("Liczba etykietowanych:", np.sum(semi_targets != -1))
    print("Liczba nieetykietowanych:", np.sum(semi_targets == -1))

    return semi_targets, labeled_indices

semi_targets, labeled_indices = get_semi_targets(targets=targets, labeled_ratio = 0.05)

Liczba etykietowanych: 89
Liczba nieetykietowanych: 1708


<div>

### Podpunkt B

</div>

In [22]:
def get_labels_from_model_1(images, semi_targets):
    kmeans = KMeans(n_clusters=10, random_state=42)
    labels = kmeans.fit_predict(images)

    propagated_labels = np.copy(semi_targets)

    for cluster_id in range(10):
        cluster_points = np.where(labels == cluster_id)[0]  # punkty w klastrze

        # punkty z etykietą w tym klastrze
        labeled_points = [i for i in cluster_points if semi_targets[i] != -1]

        if len(labeled_points) == 0:
            # brak etykiet w tym klastrze - pominąć
            continue

        # policz częstotliwości etykiet
        cluster_true_labels = semi_targets[labeled_points]
        most_common = np.bincount(cluster_true_labels).argmax()

        # przypisz tę etykietę wszystkim punktom w klastrze
        propagated_labels[cluster_points] = most_common

    return propagated_labels

propagated_labels_c1 = get_labels_from_model_1(images, semi_targets)

# Wynik propagacji
print("\nPrzykładowe propagowane etykiety:")
print(propagated_labels_c1[:50])




Przykładowe propagowane etykiety:
[0 1 1 3 4 9 6 7 8 9 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5 6 7 8 9 0 9 5 5 6 5 0
 9 8 9 8 4 1 7 7 3 5 1 0 0]


<div>

### Podpunkt C

</div>

In [23]:
def get_labels_from_model_2(images, semi_targets):
    kmeans = KMeans(n_clusters=10, random_state=42)
    labels = kmeans.fit_predict(images)
    centroids = kmeans.cluster_centers_

    propagated_labels = np.full_like(semi_targets, fill_value=-1)
    cluster_labels = {}  # będzie przechowywać wynikowe etykiety klastrów

    # 1. Najpierw oznacz klastry, które mają etykiety (jak majority vote)
    for cluster_id in range(10):
        cluster_points = np.where(labels == cluster_id)[0]
        labeled_points = [i for i in cluster_points if semi_targets[i] != -1]

        if len(labeled_points) == 0:
            continue

        # majority vote
        cluster_true_labels = semi_targets[labeled_points]
        most_common = np.bincount(cluster_true_labels).argmax()
        cluster_labels[cluster_id] = most_common

    # 2. Klastry bez etykiet - używamy najbliższego centroidu
    for cluster_id in range(10):
        if cluster_id in cluster_labels:
            continue

        # Odległość centroidu tego klastra do centroidów klastrów oznaczonych
        distances = []
        for labeled_cluster in cluster_labels:
            dist = np.linalg.norm(centroids[cluster_id] - centroids[labeled_cluster])
            distances.append((dist, labeled_cluster))

        # Najbliższy klaster
        _, nearest_cluster = min(distances, key=lambda x: x[0])
        cluster_labels[cluster_id] = cluster_labels[nearest_cluster]

    # 3. Przypisz etykiety do punktów na podstawie cluster_labels
    for i in range(len(semi_targets)):
        propagated_labels[i] = cluster_labels[labels[i]]

    return propagated_labels
    
propagated_labels_c2 = get_labels_from_model_2(images, semi_targets)

# Gotowe
print("Etykiety po propagacji (c2):")
print(propagated_labels_c2[:50])

Etykiety po propagacji (c2):
[0 1 1 3 4 9 6 7 8 9 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5 6 7 8 9 0 9 5 5 6 5 0
 9 8 9 8 4 1 7 7 3 5 1 0 0]


In [24]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

unlabeled_mask = semi_targets == -1

print(f'Model 1 accuracy: {accuracy_score(propagated_labels_c1[unlabeled_mask], targets[unlabeled_mask]):.02f}')
print(f'Model 2 accuracy: {accuracy_score(propagated_labels_c2[unlabeled_mask], targets[unlabeled_mask]):.02f}')

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(images[labeled_indices], targets[labeled_indices])

knn_predictions = knn.predict(images)
print(f'Trenowany na etykietach Model KNN  accuracy: {accuracy_score(knn_predictions[unlabeled_mask], targets[unlabeled_mask]):.02f}')

knn_full = KNeighborsClassifier(n_neighbors=5)
knn_full.fit(images, targets)

knn_full_pred = knn_full.predict(images)

print(f'Nadzorowany Model KNN  accuracy: {accuracy_score(knn_full_pred[unlabeled_mask], targets[unlabeled_mask]):.02f}')


Model 1 accuracy: 0.86
Model 2 accuracy: 0.86
Trenowany na etykietach Model KNN  accuracy: 0.81
Nadzorowany Model KNN  accuracy: 0.99


In [ ]:
res = []

for ratio in [0.01, 0.05, 0.10, 0.20]:
    semi_targets, labeled_indices = get_semi_targets(targets, labeled_ratio = ratio)
    
    pred_1 = get_labels_from_model_1(images, semi_targets)
    pred_2 = get_labels_from_model_2(images, semi_targets)

    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(images[labeled_indices], targets[labeled_indices])
    knn_pred = knn.predict(images)

    print(f'RATIO: {ratio}')
    print(f'Model 1 accuracy: {100 * accuracy_score(pred_1[unlabeled_mask], targets[unlabeled_mask]):.02f}%')
    print(f'Model 2 accuracy: {100 * accuracy_score(pred_2[unlabeled_mask], targets[unlabeled_mask]):.02f}%')
    print(f'Model 3 accuracy: {100 * accuracy_score(knn_pred[unlabeled_mask], targets[unlabeled_mask]):.02f}%')
    

Liczba etykietowanych: 17
Liczba nieetykietowanych: 1780
RATIO: 0.01
Model 1 accuracy: 71.43%
Model 2 accuracy: 71.60%
Model 3 accuracy: 32.85%
Liczba etykietowanych: 89
Liczba nieetykietowanych: 1708
RATIO: 0.05
Model 1 accuracy: 86.18%
Model 2 accuracy: 86.18%
Model 3 accuracy: 80.74%
Liczba etykietowanych: 179
Liczba nieetykietowanych: 1618
RATIO: 0.1
Model 1 accuracy: 86.18%
Model 2 accuracy: 86.18%
Model 3 accuracy: 91.57%
Liczba etykietowanych: 359
Liczba nieetykietowanych: 1438
RATIO: 0.2
Model 1 accuracy: 86.18%
Model 2 accuracy: 86.18%
Model 3 accuracy: 95.78%


## Zadanie 2

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n_trams = 200

# Normalne dane
temperature = rng.normal(loc=60, scale=5, size=n_trams)   # 60°C średnia
vibration = rng.normal(loc=2.0, scale=0.3, size=n_trams)  # wibracje (g)
speed = rng.normal(loc=40, scale=3, size=n_trams)         # km/h

# Wprowadzamy anomalie losowo u ~10 tramwajów
n_anomalies = 10
anomaly_idx = rng.choice(n_trams, n_anomalies, replace=False)

temperature[anomaly_idx] += rng.normal(20, 5, size=n_anomalies)  # przegrzewanie
vibration[anomaly_idx] += rng.normal(1.5, 0.2, size=n_anomalies) # drgania
speed[anomaly_idx] -= rng.normal(10, 3, size=n_anomalies)        # spadek prędkości

# Zbuduj DataFrame
df = pd.DataFrame({
    "temperature": temperature,
    "vibration": vibration,
    "speed": speed,
    "is_anomaly": [1 if i in anomaly_idx else 0 for i in range(n_trams)]
})

print(df.head())
print("\nLiczba anomalii:", df["is_anomaly"].sum())

data = np.array(df[["temperature", "vibration", "speed"]])
test = np.ravel(np.array(df[["is_anomaly"]]))


   temperature  vibration      speed  is_anomaly
0    61.523585   2.101272  39.461166           0
1    54.800079   2.422245  40.590328           0
2    63.752256   2.027175  42.461585           0
3    64.702824   2.193182  38.818776           0
4    50.244824   1.384948  41.563502           0

Liczba anomalii: 10
[[61.5235854   2.10127236 39.46116576  0.        ]
 [54.80007947  2.42224456 40.59032829  0.        ]
 [63.75225598  2.02717547 42.46158543  0.        ]
 [64.70282358  2.19318164 38.81877648  0.        ]
 [50.24482406  1.38494837 41.56350177  0.        ]
 [53.48910247  1.98538448 39.20248362  0.        ]
 [60.63920202  1.74703092 39.6473735   0.        ]
 [58.41878704  1.63435608 42.48855713  0.        ]
 [59.91599421  1.73655429 34.02081889  0.        ]
 [55.73478036  1.89976297 36.11058302  0.        ]
 [64.39698987  2.27477076 35.55344381  0.        ]
 [63.88895968  1.60208218 32.99915164  0.        ]
 [60.33015349  2.00918945 37.96520668  0.        ]
 [65.63620603  1.85474

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.neighbors import 

dbscan = DBSCAN(eps=3.0, min_samples=5)

pred_anomalies = dbscan.fit(data)

pred_anom_idx = np.where(pred_anomalies.labels_ == -1)[0]

pred = -1 * pred_anomalies.labels_



print(pred_anom_idx)
print(np.sort(anomaly_idx))

print(recall_score(test, pred))
print(precision_score(test, pred))
print(f1_score(test, pred))




[ 14  16  43  46  79 104 107 109 126 138 139 153 160]
[ 14  16  43  46  79 107 109 126 153 160]
1.0
0.7692307692307693
0.8695652173913043
